In [3]:
# 参数量公式验算
def params_5x5(C):
    return 25 * C**2

def params_two_3x3(C):
    return 18 * C**2

C = 64
print(f"5×5卷积参数量：{params_5x5(C)}")
print(f"两层串联3×3总参数量：{params_two_3x3(C)}")
print(f"堆叠3×3参数减少比例：{(1 - params_two_3x3(C)/params_5x5(C)):.2%}")

5×5卷积参数量：102400
两层串联3×3总参数量：73728
堆叠3×3参数减少比例：28.00%


In [3]:
import torch
import torch.nn as nn

def NinBlock(in_channels: int, out_channels: int, kernel_size: int, stride: int, padding: int) -> nn.Sequential:
    """
    标准NiN Block实现
    :param in_channels: 输入通道数
    :param out_channels: 输出通道数
    :param kernel_size: 首层卷积核大小
    :param stride: 首层卷积步幅
    :param padding: 首层卷积填充
    :return: nn.Sequential封装的NiN块
    """
    block = nn.Sequential(
        # 第一层常规卷积 + ReLU
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
        nn.ReLU(inplace=True),
        # 第一个1×1卷积 + ReLU
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True),
        # 第二个1×1卷积 + ReLU
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True)
    )
    return block


# ------------------- 测试代码 -------------------
if __name__ == "__main__":
    # 实例化NiN块
    nin_block = NinBlock(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0)
    print("NiN Block结构：")
    print(nin_block)

    # 构造随机输入 [N=2, C=3, H=224, W=224]
    x = torch.randn(2, 3, 224, 224)
    y = nin_block(x)
    print(f"\n输入shape: {x.shape}")
    print(f"输出shape: {y.shape}")

NiN Block结构：
Sequential(
  (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
  (1): ReLU(inplace=True)
  (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU(inplace=True)
  (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU(inplace=True)
)

输入shape: torch.Size([2, 3, 224, 224])
输出shape: torch.Size([2, 96, 54, 54])
